# Prévision du courrier postal — Jeu de données NATIONAL

Analyse exploratoire, prétraitement, ingénierie de caractéristiques et modélisation
prédictive (statistique, Machine Learning, Deep Learning) du **poids** et du **chiffre
d'affaires (CA)** des envois nationaux, avec accélération **CUDA/GPU automatique**.


In [ ]:
# Installation des dépendances (à exécuter une seule fois)
import sys, subprocess
pkgs = ["pandas", "numpy", "matplotlib", "seaborn", "plotly", "scikit-learn",
        "statsforecast", "neuralforecast", "lightgbm", "xgboost", "catboost",
        "optuna", "shap", "openpyxl", "torch"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Dépendances installées.")


In [ ]:
import warnings, os, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 5)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Bibliothèques importées avec succès.")


In [ ]:
DATASET_FILE   = 'dépôt2026_nettoyé.xlsx'
DATASET_SHEET  = 'national'
DATASET_ENGINE = 'openpyxl'
DATASET_HEADER = 2
GROUP_COL      = 'Region Depot'
GROUP_LABEL    = "région de dépôt"
GROUP_KEYWORDS = ("region", "depot")
HAS_CA         = True
TARGETS        = ["poids", "ca"]
DATASET_NAME   = "National"
OUTPUT_PREFIX  = "national"


In [ ]:
def normalize(s):
    """Normalise une chaîne : minuscules, sans accents, espaces compactés."""
    import unicodedata
    if not isinstance(s, str):
        return s
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("utf-8")
    return " ".join(s.lower().split())

def find_col(df, *keywords):
    """Trouve la colonne correspondant aux mots-clés : essaie d'abord une
    correspondance exacte (normalisée), puis une correspondance par sous-chaîne.
    Évite les faux positifs comme "ca" matchant "EDI_Cause"."""
    norm_kw = [normalize(k) for k in keywords]
    target = " ".join(norm_kw)
    for col in df.columns:
        if normalize(col) == target:
            return col
    for col in df.columns:
        nc = normalize(col)
        if all(k in nc for k in norm_kw):
            return col
    return None


## 1. Exploration et compréhension des données (EDA)

Avant toute modélisation, nous explorons le jeu de données **National** afin de :
- comprendre sa structure, sa qualité et ses anomalies,
- visualiser la distribution des variables clés (poids, chiffre d'affaires),
- analyser la dynamique temporelle (tendance, saisonnalité, jours ouvrés),
- identifier les groupes (région) les plus représentatifs.

Ces observations guideront le prétraitement et le choix des modèles de prévision.


In [ ]:
# 1.1 Chargement et aperçu général du jeu de données
raw = pd.read_excel(DATASET_FILE, sheet_name=DATASET_SHEET, engine=DATASET_ENGINE, header=DATASET_HEADER)
print(f"Dimensions : {raw.shape[0]:,} lignes × {raw.shape[1]} colonnes")
print(f"\nColonnes : {list(raw.columns)}")
print(f"\nTypes de données :")
print(raw.dtypes)
raw.head(10)


In [ ]:
# 1.2 Valeurs manquantes et doublons
missing = raw.isna().sum()
missing_pct = (missing / len(raw) * 100).round(2)
miss_df = pd.DataFrame({"manquantes": missing, "pourcentage (%)": missing_pct})
miss_df = miss_df[miss_df["manquantes"] > 0].sort_values("manquantes", ascending=False)
print(f"Lignes dupliquées : {raw.duplicated().sum():,}")
print(f"\nColonnes avec valeurs manquantes :")
display(miss_df) if not miss_df.empty else print("  → Aucune valeur manquante détectée.")

if not miss_df.empty:
    fig, ax = plt.subplots(figsize=(10, max(3, 0.4 * len(miss_df))))
    sns.barplot(x=miss_df["pourcentage (%)"], y=miss_df.index, ax=ax, color="indianred")
    ax.set_xlabel("% de valeurs manquantes"); ax.set_title(f"Valeurs manquantes — {DATASET_NAME}")
    plt.tight_layout(); plt.show()


In [ ]:
# 1.3 Distribution de la variable POIDS
poids_col = find_col(raw, "poids")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(raw[poids_col].dropna(), bins=60, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution du poids (échelle linéaire)")
axes[0].set_xlabel("Poids (kg)")

sns.histplot(raw[poids_col].dropna(), bins=60, kde=True, ax=axes[1], color="darkorange", log_scale=True)
axes[1].set_title("Distribution du poids (échelle log)")
axes[1].set_xlabel("Poids (kg, log)")
plt.tight_layout(); plt.show()

print(raw[poids_col].describe().round(2))

# Répartition par classes de poids
bins = [0, 1, 5, 10, 30, 70, np.inf]
labels = ["<1kg", "1-5kg", "5-10kg", "10-30kg", "30-70kg", ">70kg"]
classes = pd.cut(raw[poids_col], bins=bins, labels=labels, right=False).value_counts().sort_index()
fig = px.pie(values=classes.values, names=classes.index, title=f"Répartition par classe de poids — {DATASET_NAME}",
             hole=0.4, color_discrete_sequence=px.colors.sequential.Blues_r)
fig.update_layout(template="plotly_white")
fig.show()


In [ ]:
# 1.4 Distribution du chiffre d'affaires (CA)
if HAS_CA:
    ca_col = find_col(raw, "ca") or find_col(raw, "chiffre", "affaires")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(raw[ca_col].dropna(), bins=60, kde=True, ax=axes[0], color="seagreen")
    axes[0].set_title("Distribution du CA (échelle linéaire)"); axes[0].set_xlabel("CA")

    sns.histplot(raw[ca_col].dropna(), bins=60, kde=True, ax=axes[1], color="mediumvioletred", log_scale=True)
    axes[1].set_title("Distribution du CA (échelle log)"); axes[1].set_xlabel("CA (log)")
    plt.tight_layout(); plt.show()
    print(raw[ca_col].describe().round(2))
else:
    print("→ Ce jeu de données ne contient pas de colonne CA — section ignorée.")


In [ ]:
# 1.5 Top 20 des groupes (région) les plus fréquents
grp_col = GROUP_COL if GROUP_COL in raw.columns else find_col(raw, *GROUP_KEYWORDS)
top_groups = raw[grp_col].value_counts().head(20)

fig = px.bar(x=top_groups.values, y=top_groups.index.astype(str), orientation="h",
             title=f"Top 20 — {GROUP_LABEL} les plus fréquents ({DATASET_NAME})",
             labels={"x": "Nombre d'enregistrements", "y": GROUP_LABEL},
             color=top_groups.values, color_continuous_scale="Tealgrn")
fig.update_layout(yaxis=dict(autorange="reversed"), template="plotly_white", height=600)
fig.show()


In [ ]:
# 1.6 Relation Poids / CA
if HAS_CA:
    poids_col = find_col(raw, "poids")
    ca_col = find_col(raw, "ca") or find_col(raw, "chiffre", "affaires")
    sample = raw[[poids_col, ca_col]].dropna().sample(min(5000, len(raw)), random_state=RANDOM_STATE)
    fig = px.scatter(sample, x=poids_col, y=ca_col, opacity=0.4, trendline="ols",
                     title=f"Poids vs CA — {DATASET_NAME} (échantillon de {len(sample):,} pts)",
                     color_discrete_sequence=["royalblue"])
    fig.update_layout(template="plotly_white")
    fig.show()
    corr = raw[[poids_col, ca_col]].corr().iloc[0, 1]
    print(f"Corrélation linéaire Poids/CA : {corr:.3f}")
else:
    print("→ Pas de colonne CA disponible — relation Poids/CA non applicable.")


In [ ]:
# 1.7 Série temporelle journalière agrégée
date_col = find_col(raw, "date")
poids_col = find_col(raw, "poids")
daily = raw.groupby(pd.to_datetime(raw[date_col]).dt.date).agg(**{"poids_total": (poids_col, "sum"),
                                                                     "nb_envois": (poids_col, "count")})
daily.index = pd.to_datetime(daily.index)
daily = daily.sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily.index, daily["poids_total"], lw=0.8, alpha=0.5, color="gray", label="Poids total quotidien")
ax.plot(daily.index, daily["poids_total"].rolling(7).mean(), lw=2, color="crimson", label="Moyenne mobile 7 jours")
ax.set_title(f"Évolution du poids total quotidien — {DATASET_NAME}")
ax.set_ylabel("Poids total (kg)"); ax.legend()
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
plt.tight_layout(); plt.show()

print(f"Période couverte : {daily.index.min().date()} → {daily.index.max().date()}  ({len(daily)} jours)")


In [ ]:
# 1.8 Effet du jour de la semaine
date_col = find_col(raw, "date")
poids_col = find_col(raw, "poids")
tmp = raw[[date_col, poids_col]].dropna().copy()
tmp["jour"] = pd.to_datetime(tmp[date_col]).dt.day_name()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
fr = {"Monday": "Lundi", "Tuesday": "Mardi", "Wednesday": "Mercredi", "Thursday": "Jeudi",
       "Friday": "Vendredi", "Saturday": "Samedi", "Sunday": "Dimanche"}
tmp["jour_fr"] = tmp["jour"].map(fr)
order_fr = [fr[d] for d in order]

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=tmp, x="jour_fr", y=poids_col, order=order_fr, ax=ax, palette="coolwarm", showfliers=False)
ax.set_title(f"Distribution du poids par jour de la semaine — {DATASET_NAME}")
ax.set_xlabel(""); ax.set_ylabel("Poids (kg)")
plt.tight_layout(); plt.show()


In [ ]:
# 1.9 Volumétrie mensuelle
date_col = find_col(raw, "date")
poids_col = find_col(raw, "poids")
tmp = raw[[date_col, poids_col]].dropna().copy()
tmp["mois"] = pd.to_datetime(tmp[date_col]).dt.to_period("M").astype(str)
monthly = tmp.groupby("mois")[poids_col].agg(["sum", "count"]).reset_index()

fig = go.Figure()
fig.add_bar(x=monthly["mois"], y=monthly["sum"], name="Poids total (kg)", marker_color="steelblue")
fig.add_scatter(x=monthly["mois"], y=monthly["count"], name="Nb envois", yaxis="y2",
                mode="lines+markers", line=dict(color="firebrick"))
fig.update_layout(title=f"Volumétrie mensuelle — {DATASET_NAME}",
                  yaxis=dict(title="Poids total (kg)"),
                  yaxis2=dict(title="Nombre d'envois", overlaying="y", side="right"),
                  template="plotly_white", legend=dict(orientation="h", y=1.1))
fig.show()


In [ ]:
# 1.10 Évolution temporelle des 6 principaux groupes (région)
date_col = find_col(raw, "date")
poids_col = find_col(raw, "poids")
grp_col = GROUP_COL if GROUP_COL in raw.columns else find_col(raw, *GROUP_KEYWORDS)
top6 = raw[grp_col].value_counts().head(6).index
sub = raw[raw[grp_col].isin(top6)][[date_col, grp_col, poids_col]].dropna().copy()
sub["mois"] = pd.to_datetime(sub[date_col]).dt.to_period("M").dt.to_timestamp()
pivot = sub.groupby(["mois", grp_col])[poids_col].sum().reset_index()

fig = px.line(pivot, x="mois", y=poids_col, color=grp_col, markers=True,
              title=f"Évolution mensuelle du poids — Top 6 {GROUP_LABEL} ({DATASET_NAME})",
              labels={poids_col: "Poids total (kg)", "mois": "Mois", grp_col: GROUP_LABEL})
fig.update_layout(template="plotly_white")
fig.show()


In [ ]:
# 1.11 Carte de chaleur Groupe × Mois (intensité du volume)
date_col = find_col(raw, "date")
poids_col = find_col(raw, "poids")
grp_col = GROUP_COL if GROUP_COL in raw.columns else find_col(raw, *GROUP_KEYWORDS)
top10 = raw[grp_col].value_counts().head(10).index
sub = raw[raw[grp_col].isin(top10)][[date_col, grp_col, poids_col]].dropna().copy()
sub["mois"] = pd.to_datetime(sub[date_col]).dt.to_period("M").astype(str)
mat = sub.pivot_table(index=grp_col, columns="mois", values=poids_col, aggfunc="sum", fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(mat, cmap="YlOrRd", linewidths=0.4, ax=ax, cbar_kws={"label": "Poids total (kg)"})
ax.set_title(f"Intensité du volume par {GROUP_LABEL} et par mois — {DATASET_NAME} (Top 10)")
ax.set_xlabel("Mois"); ax.set_ylabel(GROUP_LABEL)
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()


In [ ]:
# 1.12 Analyse des délais de livraison (si disponible)
date_dep = find_col(raw, "date", "depot") or find_col(raw, "date", "depart")
date_arr = find_col(raw, "date", "arrivee") or find_col(raw, "date", "livraison")
if date_dep and date_arr:
    tmp = raw[[date_dep, date_arr]].dropna().copy()
    tmp[date_dep] = pd.to_datetime(tmp[date_dep], errors="coerce")
    tmp[date_arr] = pd.to_datetime(tmp[date_arr], errors="coerce")
    tmp = tmp.dropna()
    tmp["delai_j"] = (tmp[date_arr] - tmp[date_dep]).dt.days
    tmp = tmp[(tmp["delai_j"] >= 0) & (tmp["delai_j"] <= 30)]

    if not tmp.empty:
        fig, ax = plt.subplots(figsize=(12, 5))
        sns.ecdfplot(tmp["delai_j"], ax=ax, color="darkslateblue", lw=2)
        ax.set_title(f"Fonction de répartition (CDF) du délai de livraison — {DATASET_NAME}")
        ax.set_xlabel("Délai (jours)"); ax.set_ylabel("Proportion cumulée")
        ax.axvline(tmp["delai_j"].median(), color="crimson", ls="--", label=f"Médiane = {tmp['delai_j'].median():.0f} j")
        ax.legend(); plt.tight_layout(); plt.show()
        print(tmp["delai_j"].describe().round(2))
    else:
        print("→ Aucune date de livraison exploitable (valeurs non renseignées) — analyse des délais ignorée.")
else:
    print("→ Colonnes de dates de dépôt/arrivée introuvables — analyse des délais ignorée.")


In [ ]:
# 1.13 Répartition des événements EDI (si disponible)
edi_col = find_col(raw, "edi") or find_col(raw, "evenement")
if edi_col:
    counts = raw[edi_col].value_counts().head(15)
    fig = px.bar(x=counts.index.astype(str), y=counts.values,
                 title=f"Top 15 des événements EDI — {DATASET_NAME}",
                 labels={"x": "Événement", "y": "Occurrences"}, color=counts.values,
                 color_continuous_scale="Plasma")
    fig.update_layout(template="plotly_white", xaxis_tickangle=-40)
    fig.show()
else:
    print("→ Aucune colonne d'événement EDI détectée — section ignorée.")


In [ ]:
# 1.14 Matrice de corrélation des variables numériques
num = raw.select_dtypes(include=[np.number])
if num.shape[1] >= 2:
    corr = num.corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True,
                linewidths=0.5, cbar_kws={"shrink": 0.8})
    ax.set_title(f"Corrélations entre variables numériques — {DATASET_NAME}")
    plt.tight_layout(); plt.show()
else:
    print("→ Pas assez de variables numériques pour une matrice de corrélation.")


## 2. Prétraitement des données

Sur la base des observations précédentes, nous construisons des séries temporelles
journalières (fréquence jours ouvrés `B`) agrégées par cible (poids, ca),
en gérant les valeurs manquantes, les doublons et le filtrage du groupe **région**.


In [ ]:
def preprocess():
    """Charge, nettoie et agrège les données en séries temporelles journalières (jours ouvrés)."""
    df = pd.read_excel(DATASET_FILE, sheet_name=DATASET_SHEET, engine=DATASET_ENGINE, header=DATASET_HEADER)
    df.columns = [c.strip() for c in df.columns]

    date_col = find_col(df, "date")
    poids_col = find_col(df, "poids")
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col, poids_col])
    df = df.drop_duplicates()

    agg = {"poids": (poids_col, "sum")}
    if HAS_CA:
        ca_col = find_col(df, "ca") or find_col(df, "chiffre", "affaires")
        agg["ca"] = (ca_col, "sum")

    daily = df.groupby(df[date_col].dt.normalize()).agg(**agg).sort_index()
    full_idx = pd.date_range(daily.index.min(), daily.index.max(), freq="B")
    daily = daily.reindex(full_idx).fillna(0.0)
    daily.index.name = "ds"
    return daily

series = preprocess()
series.to_csv(f"{OUTPUT_PREFIX}_series_daily.csv")
print(f"Série journalière construite : {series.shape[0]} jours ouvrés, du {series.index.min().date()} au {series.index.max().date()}")
series.head()


In [ ]:
# Aperçu de la série prétraitée (jours ouvrés)
fig, axes = plt.subplots(len(TARGETS), 1, figsize=(14, 4 * len(TARGETS)), sharex=True)
axes = np.atleast_1d(axes)
for ax, col in zip(axes, TARGETS):
    ax.plot(series.index, series[col], lw=0.8, color="gray", alpha=0.6, label=f"{col} (quotidien)")
    ax.plot(series.index, series[col].rolling(20).mean(), lw=2, color="crimson", label="Moyenne mobile 20j")
    ax.set_title(f"Série prétraitée — {col} ({DATASET_NAME})"); ax.legend()
plt.tight_layout(); plt.show()


## 3. Ingénierie des caractéristiques (Feature Engineering)

Nous enrichissons la série avec des variables calendaires, des décalages (*lags*) et des
statistiques glissantes, utilisées par les modèles de Machine Learning (LightGBM, XGBoost, CatBoost).


In [ ]:
def add_features(s, target):
    """Ajoute des variables calendaires, lags et rolling stats pour la cible donnée."""
    out = pd.DataFrame(index=s.index)
    out["y"] = s[target]
    out["dayofweek"] = out.index.dayofweek
    out["day"] = out.index.day
    out["month"] = out.index.month
    out["quarter"] = out.index.quarter
    out["weekofyear"] = out.index.isocalendar().week.astype(int)
    out["is_month_start"] = out.index.is_month_start.astype(int)
    out["is_month_end"] = out.index.is_month_end.astype(int)

    for lag in [1, 2, 3, 5, 10, 20]:
        out[f"lag_{lag}"] = out["y"].shift(lag)
    for win in [5, 10, 20]:
        out[f"roll_mean_{win}"] = out["y"].shift(1).rolling(win).mean()
        out[f"roll_std_{win}"] = out["y"].shift(1).rolling(win).std()

    return out.dropna()


## 4. Pipeline de prévision et comparaison des modèles

Nous entraînons et comparons plusieurs familles de modèles :
- **Statistiques** : AutoARIMA, AutoTheta (StatsForecast)
- **Machine Learning** : LightGBM (+Optuna), XGBoost, CatBoost
- **Deep Learning** : N-HiTS, PatchTST, LSTM, TFT (NeuralForecast)

L'évaluation se fait via le **SMAPE** sur un ensemble de test (horizon = 30 jours ouvrés).


In [ ]:
HORIZON     = 30
HORIZON_FUT = 60
MIN_OBS     = 20
RUN_NEURAL  = True

# ── Répartition chronologique Train / Validation / Test ───────────────────
TRAIN_RATIO = 0.75
VAL_RATIO   = 0.15
TEST_RATIO  = round(1 - TRAIN_RATIO - VAL_RATIO, 2)
print(f"Répartition des données : train={TRAIN_RATIO:.0%} · validation={VAL_RATIO:.0%} · test={TEST_RATIO:.0%}")

# ── Détection GPU / CUDA ───────────────────────────────────────────────────
try:
    import torch as _torch
    USE_GPU = _torch.cuda.is_available()
    if USE_GPU:
        print(f"GPU détecté : {_torch.cuda.get_device_name(0)} — CUDA activé "
              f"(XGBoost · CatBoost · NeuralForecast)")
    else:
        print("Aucun GPU CUDA détecté — exécution sur CPU")
except ImportError:
    USE_GPU = False
    print("PyTorch indisponible — USE_GPU=False (exécution sur CPU)")

def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred))
    diff = np.abs(y_true - y_pred) / np.where(denom == 0, 1, denom)
    return 200.0 * np.mean(diff)

def compute_metrics(y_true, y_pred):
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    return {
        "MAE":   mean_absolute_error(y_true, y_pred),
        "RMSE":  mean_squared_error(y_true, y_pred) ** 0.5,
        "SMAPE": smape(y_true, y_pred),
    }

def to_nixtla(s, target):
    df = pd.DataFrame({"unique_id": target, "ds": s.index, "y": s[target].values})
    return df

def train_test_nixtla(df, horizon):
    return df.iloc[:-horizon].copy(), df.iloc[-horizon:].copy()

print(f"Configuration : horizon={HORIZON}j, horizon futur={HORIZON_FUT}j, GPU={USE_GPU}")

# ── Récapitulatif : CUDA est-il actif pour chaque famille de modèles ? ─────
print("Disponibilité CUDA par modèle :")
print(f"  • StatsForecast (AutoARIMA/AutoTheta) : CPU uniquement (pas de support GPU)")
print(f"  • LightGBM                            : CPU uniquement (build GPU indisponible sous Windows/pip)")
print(f"  • XGBoost                             : {'GPU (CUDA) actif' if USE_GPU else 'CPU (CUDA indisponible)'}")
print(f"  • CatBoost                            : {'GPU (CUDA) actif' if USE_GPU else 'CPU (CUDA indisponible)'}")
print(f"  • NeuralForecast (NHITS/PatchTST/...) : {'GPU (CUDA) actif' if USE_GPU else 'CPU (CUDA indisponible)'}")


In [ ]:
def run_statsforecast(train, test, horizon):
    """Entraîne AutoARIMA et AutoTheta (StatsForecast) et retourne prévisions + métriques."""
    from statsforecast import StatsForecast
    from statsforecast.models import AutoARIMA, AutoTheta

    sf = StatsForecast(models=[AutoARIMA(season_length=5), AutoTheta(season_length=5)],
                       freq="B", n_jobs=-1)
    sf.fit(train[["unique_id", "ds", "y"]])
    fcst = sf.predict(h=horizon)
    fcst = fcst.merge(test[["ds", "y"]], on="ds", how="left")

    results = {}
    for model in ["AutoARIMA", "AutoTheta"]:
        if model in fcst.columns:
            results[model] = {"forecast": fcst[["ds", model]].rename(columns={model: "yhat"}),
                               "metrics": compute_metrics(fcst["y"], fcst[model])}
    return results


In [ ]:
# Prophet est volontairement omis (dépendance lourde, peu fiable sous Windows) :
# StatsForecast (AutoARIMA / AutoTheta) couvre déjà la famille des modèles statistiques.
PROPHET_AVAILABLE = False


In [ ]:
def lgb_objective(trial, X_train, y_train, X_val, y_val):
    import lightgbm as lgb
    params = {
        "objective": "regression", "metric": "mae", "verbosity": -1,
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": RANDOM_STATE,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False)])
    pred = model.predict(X_val)
    return compute_metrics(y_val, pred)["MAE"]


def run_ml_models(s, target):
    """Entraîne LightGBM (+Optuna), XGBoost et CatBoost avec un split chronologique
    Train / Validation / Test (proportions définies par TRAIN_RATIO / VAL_RATIO / TEST_RATIO)."""
    import lightgbm as lgb
    import xgboost as xgb
    import catboost as cb
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    feat = add_features(s, target)
    n = len(feat)
    if n <= MIN_OBS * 3:
        return {}

    # ── Split chronologique Train / Validation / Test (ordre temporel préservé) ──
    train_end = max(MIN_OBS, int(n * TRAIN_RATIO))
    val_end   = min(n - 1, max(train_end + 1, int(n * (TRAIN_RATIO + VAL_RATIO))))

    X, y = feat.drop(columns=["y"]), feat["y"]
    X_tr,  y_tr  = X.iloc[:train_end],        y.iloc[:train_end]
    X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
    X_te,  y_te  = X.iloc[val_end:],          y.iloc[val_end:]
    X_fit, y_fit = pd.concat([X_tr, X_val]),  pd.concat([y_tr, y_val])

    print(f"  Split chronologique → train={len(X_tr)} ({len(X_tr)/n:.0%}) · "
          f"validation={len(X_val)} ({len(X_val)/n:.0%}) · test={len(X_te)} ({len(X_te)/n:.0%})")

    results = {}

    # ---- LightGBM + Optuna  →  device : CPU (pas de build GPU sous Windows/pip) ----
    print("  [GPU check] LightGBM  → device = CPU  (build GPU indisponible sous Windows/pip)")
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(lambda t: lgb_objective(t, X_tr, y_tr, X_val, y_val), n_trials=15, show_progress_bar=False)
    best_params = {**study.best_params, "objective": "regression", "verbosity": -1, "random_state": RANDOM_STATE}
    lgb_m = lgb.LGBMRegressor(**best_params)
    lgb_m.fit(X_fit, y_fit)
    pred = lgb_m.predict(X_te)
    results["LightGBM"] = {"model": lgb_m,
                           "forecast": pd.DataFrame({"ds": s.index[val_end:], "yhat": pred}),
                           "metrics": compute_metrics(y_te, pred)}

    # ---- XGBoost  →  device : GPU (CUDA) si USE_GPU, sinon CPU ----
    print(f"  [GPU check] XGBoost   → device = {'GPU (CUDA) — actif' if USE_GPU else 'CPU — CUDA non disponible'}")
    xgb_m = xgb.XGBRegressor(n_estimators=800, max_depth=6, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             reg_alpha=0.1, reg_lambda=1.0,
                             tree_method="hist", device=("cuda" if USE_GPU else "cpu"),
                             random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    xgb_m.fit(X_fit, y_fit)
    pred = xgb_m.predict(X_te)
    results["XGBoost"] = {"model": xgb_m,
                          "forecast": pd.DataFrame({"ds": s.index[val_end:], "yhat": pred}),
                          "metrics": compute_metrics(y_te, pred)}

    # ---- CatBoost  →  device : GPU (CUDA) si USE_GPU, sinon CPU ----
    print(f"  [GPU check] CatBoost  → device = {'GPU (CUDA) — actif' if USE_GPU else 'CPU — CUDA non disponible'}")
    cat_m = cb.CatBoostRegressor(iterations=800, depth=6, learning_rate=0.05,
                                 loss_function="RMSE", random_seed=RANDOM_STATE, verbose=0,
                                 task_type=("GPU" if USE_GPU else "CPU"))
    cat_m.fit(X_fit, y_fit)
    pred = cat_m.predict(X_te)
    results["CatBoost"] = {"model": cat_m,
                           "forecast": pd.DataFrame({"ds": s.index[val_end:], "yhat": pred}),
                           "metrics": compute_metrics(y_te, pred)}

    return results


In [ ]:
def run_neuralforecast(train, test, horizon):
    """Entraîne N-HiTS, PatchTST, LSTM et TFT (NeuralForecast) — CUDA auto-activé si disponible."""
    if not RUN_NEURAL or len(train) < horizon * 3:
        return {}

    from neuralforecast import NeuralForecast
    from neuralforecast.models import NHITS, PatchTST, LSTM, TFT

    print(f"  [GPU check] NeuralForecast → device = "
          f"{'GPU (CUDA, num_gpus=1) — actif' if USE_GPU else 'CPU — CUDA non disponible'}")

    common = dict(h=horizon, input_size=2 * horizon, max_steps=200, random_seed=RANDOM_STATE)
    models = [
        NHITS(**common),
        PatchTST(**common),
        LSTM(**common),
        TFT(**common),
    ]

    try:
        nf = NeuralForecast(models=models, freq="B", num_gpus=int(USE_GPU))
    except TypeError:
        nf = NeuralForecast(models=models, freq="B")

    nf.fit(df=train[["unique_id", "ds", "y"]])
    fcst = nf.predict()
    fcst = fcst.merge(test[["ds", "y"]], on="ds", how="left")

    results = {}
    for model in ["NHITS", "PatchTST", "LSTM", "TFT"]:
        if model in fcst.columns:
            results[model] = {"forecast": fcst[["ds", model]].rename(columns={model: "yhat"}),
                              "metrics": compute_metrics(fcst["y"], fcst[model])}
    return results


In [ ]:
# 4.1 Préparation des données (split Nixtla par cible)
splits = {}
for target in TARGETS:
    nx = to_nixtla(series, target)
    train, test = train_test_nixtla(nx, HORIZON)
    splits[target] = (train, test)
print("Données préparées pour les cibles :", ", ".join(TARGETS))


In [ ]:
# 4.2 Modèles statistiques (StatsForecast — AutoARIMA / AutoTheta)  ⚡ rapide
stats_results = {}
for target in TARGETS:
    train, test = splits[target]
    print(f"\n→ [Statistiques] cible = {target.upper()}  —  {DATASET_NAME}")
    stats_results[target] = run_statsforecast(train, test, HORIZON)
print("\nTerminé : modèles statistiques.")


In [ ]:
# 4.3 Modèles Machine Learning (LightGBM + Optuna · XGBoost · CatBoost)  ⏱ moyen
ml_results = {}
for target in TARGETS:
    print(f"\n→ [Machine Learning] cible = {target.upper()}  —  {DATASET_NAME}")
    ml_results[target] = run_ml_models(series, target)
print("\nTerminé : modèles Machine Learning.")


In [ ]:
# 4.4 Modèles Deep Learning (NeuralForecast — NHITS · PatchTST · LSTM · TFT)  🐢 lent
# → Pour ignorer cette étape (la plus coûteuse), repasser RUN_NEURAL = False dans la cellule de configuration.
neural_results = {}
if RUN_NEURAL:
    for target in TARGETS:
        train, test = splits[target]
        print(f"\n→ [Deep Learning] cible = {target.upper()}  —  {DATASET_NAME}")
        neural_results[target] = run_neuralforecast(train, test, HORIZON)
    print("\nTerminé : modèles Deep Learning.")
else:
    neural_results = {target: {} for target in TARGETS}
    print("RUN_NEURAL = False → modèles Deep Learning ignorés.")


In [ ]:
# 4.5 Consolidation des résultats (toutes familles de modèles confondues)
all_runs = []
for target in TARGETS:
    train, test = splits[target]
    combined = {**stats_results.get(target, {}),
                **ml_results.get(target, {}),
                **neural_results.get(target, {})}
    all_runs.append((target, train, test, combined))

print(f"{len(all_runs)} cible(s) consolidée(s) : {', '.join(TARGETS)}  "
      f"({sum(len(r[3]) for r in all_runs)} modèles au total)")


## 5. Comparaison des modèles

In [ ]:
# Tableau comparatif des métriques (toutes cibles confondues)
rows = []
for target, train, test, res in all_runs:
    for model, r in res.items():
        rows.append({"cible": target, "modèle": model, **r["metrics"]})
results_df = pd.DataFrame(rows).sort_values(["cible", "SMAPE"]).reset_index(drop=True)
results_df.style.background_gradient(subset=["SMAPE"], cmap="RdYlGn_r")


In [ ]:
# Comparaison visuelle du SMAPE par modèle et par cible
fig = px.bar(results_df, x="modèle", y="SMAPE", color="cible", barmode="group",
             title=f"Comparaison du SMAPE par modèle — {DATASET_NAME}",
             text_auto=".1f")
fig.update_layout(template="plotly_white", legend=dict(orientation="h", y=1.1))
fig.show()


In [ ]:
# Vue radar des performances normalisées (1 = meilleur modèle par métrique)
metrics_cols = ["MAE", "RMSE", "SMAPE"]
radar_df = results_df.copy()
for col in metrics_cols:
    radar_df[col] = radar_df.groupby("cible")[col].transform(lambda x: x.min() / x)

fig = go.Figure()
for model in radar_df["modèle"].unique():
    sub = radar_df[radar_df["modèle"] == model][metrics_cols].mean()
    fig.add_trace(go.Scatterpolar(r=sub.values, theta=metrics_cols, fill="toself", name=model))
fig.update_layout(title=f"Performance relative des modèles — {DATASET_NAME} (1 = meilleur)",
                  polar=dict(radialaxis=dict(visible=True, range=[0, 1])), template="plotly_white")
fig.show()


## 5. Explicabilité (SHAP)

Nous utilisons **SHAP** pour interpréter les prévisions du meilleur modèle ML (LightGBM/XGBoost/CatBoost)
et identifier les variables les plus influentes.


In [ ]:
def run_shap(target):
    """Calcule et affiche les valeurs SHAP pour le meilleur modèle ML entraîné sur `target`."""
    import shap
    feat = add_features(series, target)
    X = feat.drop(columns=["y"])
    X_sample = X.sample(min(500, len(X)), random_state=RANDOM_STATE)

    import lightgbm as lgb
    model = lgb.LGBMRegressor(n_estimators=400, random_state=RANDOM_STATE, verbosity=-1)
    model.fit(X, feat["y"])

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

    shap.summary_plot(shap_values, X_sample, show=False, plot_size=(12, 6))
    plt.title(f"Importance des variables (SHAP) — {target} — {DATASET_NAME}")
    plt.tight_layout(); plt.show()


In [ ]:
for target in TARGETS:
    run_shap(target)


## 6. Prévision future (60 jours ouvrés)

Nous projetons enfin l'évolution future de chaque cible au-delà des données disponibles,
en ré-entraînant le meilleur modèle sur l'intégralité de l'historique.


In [ ]:
def forecast_future(target, horizon=HORIZON_FUT):
    """Ré-entraîne AutoARIMA sur tout l'historique et prévoit `horizon` jours ouvrés futurs."""
    from statsforecast import StatsForecast
    from statsforecast.models import AutoARIMA

    nx = to_nixtla(series, target)
    sf = StatsForecast(models=[AutoARIMA(season_length=5)], freq="B", n_jobs=-1)
    sf.fit(nx[["unique_id", "ds", "y"]])
    future = sf.predict(h=horizon)
    return future.rename(columns={"AutoARIMA": "yhat"})


def plot_future(target, future):
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(series.index[-180:], series[target].iloc[-180:], color="steelblue", label="Historique (6 derniers mois)")
    ax.plot(future["ds"], future["yhat"], color="crimson", ls="--", marker="o", ms=3, label=f"Prévision +{HORIZON_FUT}j")
    ax.axvline(series.index[-1], color="gray", ls=":", lw=1)
    ax.set_title(f"Prévision future — {target} — {DATASET_NAME}")
    ax.legend(); plt.tight_layout(); plt.show()


def plot_global_forecast(futures):
    fig = go.Figure()
    for target, future in futures.items():
        fig.add_scatter(x=series.index[-90:], y=series[target].iloc[-90:], mode="lines",
                        name=f"{target} (historique)", line=dict(width=1))
        fig.add_scatter(x=future["ds"], y=future["yhat"], mode="lines+markers",
                        name=f"{target} (prévision +{HORIZON_FUT}j)", line=dict(dash="dash"))
    fig.update_layout(title=f"Vue globale des prévisions futures — {DATASET_NAME}",
                      template="plotly_white", legend=dict(orientation="h", y=1.15))
    fig.show()


In [ ]:
futures = {}
for target in TARGETS:
    print(f"→ Prévision future pour '{target}'...")
    fut = forecast_future(target)
    fut.to_csv(f"{OUTPUT_PREFIX}_future_{target}.csv", index=False)
    futures[target] = fut
    plot_future(target, fut)


In [ ]:
plot_global_forecast(futures)


## 7. Synthèse

Ce notebook a couvert, pour le jeu de données **National** :
1. L'exploration et la compréhension des données (EDA) — distributions, séries temporelles, groupes, corrélations.
2. Le prétraitement en séries journalières (jours ouvrés) et l'ingénierie de caractéristiques.
3. L'entraînement et la comparaison de modèles statistiques, ML et deep learning (accélération **CUDA/GPU automatique** pour XGBoost, CatBoost et NeuralForecast lorsqu'un GPU NVIDIA est disponible).
4. L'explicabilité via SHAP.
5. La prévision future à 60 jours ouvrés.

**Fichiers générés :**
- `national_series_daily.csv` — série journalière prétraitée
- `national_future_<cible>.csv` — prévisions futures par cible
